In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# OSL Process Matching — Tam Notebook
# Filtre : Lead Table = Q2E, Process seviyesi = L2
# Doküman: /Volumes/hcdap_prod/bronze_hc_tri_rlc/alevsplace/all_docs_combined.txt
# ═══════════════════════════════════════════════════════════════════════════════


# ─── ADIM 1: Template süreçleri yükle (Q2E + L2 filtreli) ────────────────────
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

raw_df = spark.sql("""
    SELECT
        `Process ID`          AS process_id,
        `Process Name`        AS process_name,
        `Process Description` AS process_desc,
        `Lead Table`          AS lead_table,
        `Process Type`        AS process_type,
        `Data Status`         AS data_status
    FROM hcdap_prod.bronze_hc_tri_rlc.osl_export_290526
    WHERE `Data Status` = 'Productive'
      AND `Lead Table`  = 'Q2E'
""").toPandas()

print(f"Q2E toplam süreç: {len(raw_df)}")

# L2 seviyesi = Process ID'de nokta sayısı 2 olan (CPT.Q2E.xxx formatı)
# Yani split('.') ile 3 parça olanlar L2, daha fazlası L3/L4
def get_level(pid):
    parts = str(pid).split('.')
    return len(parts)  # 3=L2, 4=L3, 5=L4

raw_df['level'] = raw_df['process_id'].apply(get_level)

print("\nSeviye dağılımı (nokta sayısı):")
print(raw_df['level'].value_counts().sort_index().to_string())

# L2 = en kısa process ID'ler (3 segment: CPT.Q2E.010)
# Eğer Q2E'de hiyerarşi farklıysa aşağıdaki sayıyı ayarla
L2_LEVEL = raw_df['level'].min()  # en üst seviye = L2
template_df = raw_df[raw_df['level'] == L2_LEVEL].copy().reset_index(drop=True)

print(f"\nL2 süreç sayısı (level={L2_LEVEL}): {len(template_df)}")
print("\nÖrnek L2 süreçler:")
for _, r in template_df.head(5).iterrows():
    print(f"  {r['process_id']} | {r['process_name']}")


# ─── ADIM 2: Arama metni oluştur ─────────────────────────────────────────────
def build_search_text(row):
    parts = []
    for col in ['process_name', 'process_desc']:
        val = str(row.get(col, '')).strip()
        if val and val.lower() not in ['nan', 'none', 'tbd', '-', '']:
            parts.append(val)
    return ' '.join(parts)

template_df['search_text'] = template_df.apply(build_search_text, axis=1)

print("\nÖrnek search_text:")
for _, r in template_df.head(3).iterrows():
    print(f"  [{r['process_id']}] {r['search_text'][:100]}")


# ─── ADIM 3: Dokümanları oku ve chunk'lara böl ───────────────────────────────
DOC_PATH = "/Volumes/hcdap_prod/bronze_hc_tri_rlc/alevsplace/all_docs_combined.txt"

with open(DOC_PATH, 'r', encoding='utf-8') as f:
    full_text = f.read()

print(f"\nDoküman okundu: {len(full_text.split())} kelime")

# 300 kelimelik chunk'lara böl
words = full_text.split()
doc_records = []
for i in range(0, len(words), 300):
    chunk = ' '.join(words[i:i+300])
    doc_records.append({
        'source_file': 'all_docs_combined.txt',
        'chunk_id':    i // 300,
        'chunk_text':  chunk
    })

print(f"Chunk sayısı: {len(doc_records)}")


# ─── ADIM 4: TF-IDF Matching ─────────────────────────────────────────────────
docs_df        = pd.DataFrame(doc_records)
template_texts = template_df['search_text'].tolist()
doc_texts      = docs_df['chunk_text'].tolist()

vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=0.90,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix    = vectorizer.fit_transform(template_texts + doc_texts)
template_matrix = tfidf_matrix[:len(template_texts)]
doc_matrix      = tfidf_matrix[len(template_texts):]

print(f"Vocabulary: {len(vectorizer.vocabulary_)} kelime")

THRESHOLD = 0.05   # Q2E süreçleri az olduğu için eşiği düşük tuttuk
TOP_K     = 5      # Her süreç için en iyi 5 eşleşme

results = []
for i, t_row in template_df.iterrows():
    scores  = cosine_similarity(template_matrix[i], doc_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_idx):
        score = scores[idx]
        if score >= THRESHOLD:
            d_row = docs_df.iloc[idx]
            results.append({
                'process_id':       t_row['process_id'],
                'process_name':     t_row['process_name'],
                'lead_table':       t_row['lead_table'],
                'level':            t_row['level'],
                'match_rank':       rank + 1,
                'similarity_score': round(float(score), 4),
                'güven':            'YÜKSEK' if score >= 0.30 else 'ORTA' if score >= 0.15 else 'DÜŞÜK',
                'source_file':      d_row['source_file'],
                'matched_text':     d_row['chunk_text'][:400],
            })

results_df = pd.DataFrame(results)

print(f"\n✓ Matching tamamlandı")
print(f"  Eşleşen süreç : {results_df['process_id'].nunique() if len(results_df)>0 else 0} / {len(template_df)}")
print(f"  Toplam eşleşme: {len(results_df)}")

if len(results_df) > 0:
    print(f"\nGüven dağılımı (rank=1):")
    print(results_df[results_df['match_rank']==1]['güven'].value_counts().to_string())


# ─── ADIM 5: Sonuçları göster ────────────────────────────────────────────────
if len(results_df) > 0:
    display(
        results_df[results_df['match_rank'] == 1]
        .sort_values('similarity_score', ascending=False)
    )
else:
    print("\nEşleşme bulunamadı — THRESHOLD değerini düşürmeyi dene (şu an: 0.05)")
    print("Veya template'deki Q2E süreçlerinin search_text'lerini kontrol et:")
    display(template_df[['process_id','process_name','search_text']].head(10))


# ─── ADIM 6: Eşleşmeyen süreçleri göster ────────────────────────────────────
if len(results_df) > 0:
    matched_ids   = set(results_df['process_id'].tolist())
    unmatched_df  = template_df[~template_df['process_id'].isin(matched_ids)][['process_id','process_name']]
    print(f"\nEşleşmeyen süreçler ({len(unmatched_df)} adet):")
    display(unmatched_df)


# ─── ADIM 7: Delta Table olarak kaydet ───────────────────────────────────────
if len(results_df) > 0:
    spark.createDataFrame(results_df).write \
        .format('delta').mode('overwrite') \
        .saveAsTable('hcdap_prod.bronze_hc_tri_rlc.osl_q2e_match_results')

    print("\n✓ Kaydedildi: hcdap_prod.bronze_hc_tri_rlc.osl_q2e_match_results")
    print("  Görüntüle:")
    print("  SELECT * FROM hcdap_prod.bronze_hc_tri_rlc.osl_q2e_match_results")
    print("  ORDER BY similarity_score DESC")